<a href="https://colab.research.google.com/github/AdrionRosanelli/LoRa_Sionna/blob/main/LoRa_phy_Sionna.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Implementação camada física do LoRa

Realizada pelo Claude. (versão gerada pelo Prof Gustavo)

### Imports

Instalação e importação das bibliotecas. (Import do tutorial Part 1 Sionna PHY)

In [4]:
import os # Configure which GPU
if os.getenv("CUDA_VISIBLE_DEVICES") is None:
    gpu_num = 0 # Use "" to use the CPU
    os.environ["CUDA_VISIBLE_DEVICES"] = f"{gpu_num}"

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# Import Sionna
try:
    import sionna.phy
except ImportError as e:
    import sys
    if 'google.colab' in sys.modules:
       # Install Sionna in Google Colab
       print("Installing Sionna and restarting the runtime. Please run the cell again.")
       os.system("pip install sionna")
       os.kill(os.getpid(), 5)
    else:
       raise e

# Configure the notebook to use only a single GPU and allocate only as much memory as needed
# For more details, see https://www.tensorflow.org/guide/gpu
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.experimental.set_memory_growth(gpus[0], True)
    except RuntimeError as e:
        print(e)

# Avoid warnings from TensorFlow
tf.get_logger().setLevel('ERROR')

import numpy as np

# For plotting
%matplotlib inline
# also try %matplotlib widget

import matplotlib.pyplot as plt

# for performance measurements
import time

#Outros imports
import sionna
from sionna.phy.utils import ebnodb2no
#from sionna.phy.channel import RayleighBlockFading, OFDMChannel, AddAWGN
from sionna.phy.channel import RayleighBlockFading, OFDMChannel, AWGN
#from sionna.phy.fec.linear import LinearEncoder, LinearDecoder
from sionna.phy.fec.linear import LinearEncoder, OSDecoder
from sionna.phy.mapping import Mapper, Demapper, BinarySource
from sionna.phy.ofdm import ResourceGrid, ResourceGridMapper, LSChannelEstimator, ResourceGridDemapper
from sionna.phy.mimo import StreamManagement

### Gerador de Chirps

In [5]:
class LoRaChirpGenerator(tf.keras.layers.Layer):
    """
    Gerador de chirps para modulação LoRa CSS (Chirp Spread Spectrum)
    """
    def __init__(self, sf=7, bw=125e3, fs=1e6, **kwargs):
        super().__init__(**kwargs)
        self.sf = sf  # Spreading Factor (7-12)
        self.bw = bw  # Bandwidth (Hz)
        self.fs = fs  # Sampling frequency (Hz)
        self.n_symbols = 2**sf  # Número de símbolos possíveis
        self.t_symbol = self.n_symbols / bw  # Duração do símbolo
        self.n_samples = int(self.t_symbol * fs)  # Amostras por símbolo

        # Pré-calcular os chirps base
        self._generate_base_chirps()

    def _generate_base_chirps(self):
        """Gera os chirps base up e down"""
        t = tf.linspace(0.0, self.t_symbol, self.n_samples)

        # Up-chirp (frequência cresce linearmente)
        f_inst_up = self.bw * (t / self.t_symbol - 0.5)
        phase_up = 2 * np.pi * tf.cumsum(f_inst_up) * (self.t_symbol / self.n_samples)
        self.up_chirp = tf.cast(tf.exp(1j * phase_up), tf.complex64)

        # Down-chirp (frequência decresce linearmente)
        f_inst_down = -self.bw * (t / self.t_symbol - 0.5)
        phase_down = 2 * np.pi * tf.cumsum(f_inst_down) * (self.t_symbol / self.n_samples)
        self.down_chirp = tf.cast(tf.exp(1j * phase_down), tf.complex64)

    def call(self, symbols):
        """
        Modula símbolos em chirps
        Args:
            symbols: Tensor de símbolos inteiros [batch_size, n_symbols]
        Returns:
            chirped_signal: Sinais chirp modulados [batch_size, n_symbols, n_samples]
        """
        batch_size = tf.shape(symbols)[0]
        n_symbols = tf.shape(symbols)[1]

        # Converte símbolos para shift de frequência
        freq_shifts = tf.cast(symbols, tf.float32) / self.n_symbols

        # Gera chirps modulados
        chirped_signals = []
        for i in range(n_symbols):
            # Shift circular do up-chirp baseado no símbolo
            shift_samples = tf.cast(freq_shifts[:, i] * self.n_samples, tf.int32)
            shifted_chirp = tf.roll(self.up_chirp, shift_samples, axis=0)
            chirped_signals.append(shifted_chirp)

        return tf.stack(chirped_signals, axis=1)


### Demodulador

In [6]:
class LoRaDemodulator(tf.keras.layers.Layer):
    """
    Demodulador LoRa usando correlação com down-chirp
    """
    def __init__(self, sf=7, bw=125e3, fs=1e6, **kwargs):
        super().__init__(**kwargs)
        self.sf = sf
        self.bw = bw
        self.fs = fs
        self.n_symbols = 2**sf
        self.t_symbol = self.n_symbols / bw
        self.n_samples = int(self.t_symbol * fs)

        # Gera down-chirp para demodulação
        self._generate_down_chirp()

    def _generate_down_chirp(self):
        """Gera down-chirp para demodulação"""
        t = tf.linspace(0.0, self.t_symbol, self.n_samples)
        f_inst_down = -self.bw * (t / self.t_symbol - 0.5)
        phase_down = 2 * np.pi * tf.cumsum(f_inst_down) * (self.t_symbol / self.n_samples)
        self.down_chirp = tf.cast(tf.exp(1j * phase_down), tf.complex64)

    def call(self, received_signal):
        """
        Demodula sinais chirp recebidos
        Args:
            received_signal: [batch_size, n_symbols, n_samples]
        Returns:
            demodulated_symbols: Símbolos demodulados [batch_size, n_symbols]
        """
        batch_size = tf.shape(received_signal)[0]
        n_symbols = tf.shape(received_signal)[1]

        demodulated = []
        for i in range(n_symbols):
            # Multiplica pelo down-chirp
            dechirped = received_signal[:, i, :] * tf.conj(self.down_chirp)

            # FFT para encontrar pico de frequência
            fft_result = tf.signal.fft(dechirped)

            # Encontra índice do pico (símbolo demodulado)
            peak_idx = tf.argmax(tf.abs(fft_result), axis=-1)
            demodulated.append(peak_idx)

        return tf.stack(demodulated, axis=1)

### Forward Error Correction (FEC)

In [7]:
class LoRaFEC(tf.keras.layers.Layer):
    """
    Implementação do FEC padrão LoRa baseado em códigos Hamming
    """
    def __init__(self, cr=1, **kwargs):
        super().__init__(**kwargs)
        self.cr = cr  # Code Rate: 1=4/5, 2=4/6, 3=4/7, 4=4/8

        # Define parâmetros baseados no CR
        self.code_rates = {
            1: (4, 5),   # CR1: 4/5
            2: (4, 6),   # CR2: 4/6
            3: (4, 7),   # CR3: 4/7
            4: (4, 8)    # CR4: 4/8
        }

        self.k, self.n = self.code_rates[cr]  # k bits de informação, n bits codificados
        self.rate = self.k / self.n

        # Gera matriz geradora para código Hamming
        self._generate_hamming_matrices()

    def _generate_hamming_matrices(self):
        """Gera matrizes G (geradora) e H (verificação de paridade) para código Hamming"""
        # Para LoRa, usamos uma versão modificada baseada no CR
        parity_bits = self.n - self.k

        # Matriz identidade para bits sistemáticos
        I = np.eye(self.k, dtype=np.int32)

        # Matriz de paridade (simplificada para demonstração)
        if parity_bits == 1:  # CR1 (4,5)
            P = np.array([[1], [1], [0], [1]], dtype=np.int32)
        elif parity_bits == 2:  # CR2 (4,6)
            P = np.array([[1, 0], [0, 1], [1, 1], [1, 0]], dtype=np.int32)
        elif parity_bits == 3:  # CR3 (4,7)
            P = np.array([[1, 0, 1], [0, 1, 1], [1, 1, 0], [1, 1, 1]], dtype=np.int32)
        else:  # CR4 (4,8)
            P = np.array([[1, 0, 1, 0], [0, 1, 1, 0], [1, 1, 0, 1], [1, 0, 1, 1]], dtype=np.int32)

        # Matriz geradora G = [I | P]
        self.G = tf.constant(np.hstack([I, P]), dtype=tf.float32)

        # Matriz de verificação H = [P^T | I]
        I_parity = np.eye(parity_bits, dtype=np.int32)
        self.H = tf.constant(np.hstack([P.T, I_parity]), dtype=tf.float32)

    def encode(self, info_bits):
        """
        Codifica bits de informação usando código Hamming LoRa
        Args:
            info_bits: [batch_size, num_blocks * k]
        Returns:
            coded_bits: [batch_size, num_blocks * n]
        """
        batch_size = tf.shape(info_bits)[0]
        total_bits = tf.shape(info_bits)[1]
        num_blocks = total_bits // self.k

        # Reshape para blocos de k bits
        info_reshaped = tf.reshape(info_bits[:, :num_blocks*self.k], [batch_size, num_blocks, self.k])

        # Codificação: c = u * G (mod 2)
        coded_blocks = tf.linalg.matmul(info_reshaped, self.G)
        coded_blocks = tf.math.mod(coded_blocks, 2)

        # Reshape de volta
        coded_bits = tf.reshape(coded_blocks, [batch_size, num_blocks * self.n])

        return coded_bits

    def decode(self, received_bits):
        """
        Decodifica bits recebidos usando síndrome de Hamming
        Args:
            received_bits: [batch_size, num_blocks * n]
        Returns:
            decoded_bits: [batch_size, num_blocks * k]
        """
        batch_size = tf.shape(received_bits)[0]
        total_bits = tf.shape(received_bits)[1]
        num_blocks = total_bits // self.n

        # Reshape para blocos de n bits
        received_reshaped = tf.reshape(received_bits[:, :num_blocks*self.n], [batch_size, num_blocks, self.n])

        # Cálculo da síndrome: s = r * H^T (mod 2)
        syndrome = tf.linalg.matmul(received_reshaped, tf.transpose(self.H))
        syndrome = tf.math.mod(syndrome, 2)

        # Correção de erro simples (para demonstração)
        # Em implementação real, seria necessário lookup table para correção
        corrected_blocks = received_reshaped

        # Extrai bits de informação (primeiros k bits de cada bloco)
        decoded_blocks = corrected_blocks[:, :, :self.k]

        # Reshape de volta
        decoded_bits = tf.reshape(decoded_blocks, [batch_size, num_blocks * self.k])

        return decoded_bits

### Impolementação da camada física

In [8]:
class LoRaPHY(tf.keras.Model):
    """
    Implementação completa da camada física LoRa
    """
    def __init__(self,
                 sf=7,           # Spreading Factor
                 cr=1,           # Code Rate (1-4)
                 bw=125e3,       # Bandwidth
                 fs=1e6,         # Sampling frequency
                 **kwargs):
        super().__init__(**kwargs)

        self.sf = sf
        self.cr = cr
        self.bw = bw
        self.fs = fs
        self.n_bits_per_symbol = sf

        # Componentes da camada física
        self.binary_source = BinarySource()

        # FEC padrão LoRa (códigos Hamming)
        self.fec = LoRaFEC(cr=cr)

        # Modulação LoRa
        self.chirp_generator = LoRaChirpGenerator(sf=sf, bw=bw, fs=fs)
        self.demodulator = LoRaDemodulator(sf=sf, bw=bw, fs=fs)

        # Canal
        self.awgn_channel = AddAWGN()

    def transmitter(self, batch_size, ebno_db):
        """
        Cadeia de transmissão LoRa
        """
        # Gera bits de informação (múltiplo de k para blocos completos)
        info_bits_per_block = self.fec.k
        num_blocks = 60 // info_bits_per_block  # Ajusta para ter blocos completos
        total_info_bits = num_blocks * info_bits_per_block

        info_bits = self.binary_source([batch_size, total_info_bits])

        # Codificação FEC LoRa
        coded_bits = self.fec.encode(info_bits)

        # Padding para múltiplo de SF se necessário
        coded_length = tf.shape(coded_bits)[1]
        padding_needed = (self.sf - (coded_length % self.sf)) % self.sf
        if padding_needed > 0:
            padding = tf.zeros([batch_size, padding_needed], dtype=tf.float32)
            coded_bits = tf.concat([coded_bits, padding], axis=1)

        # Converte bits para símbolos LoRa (agrupamento de SF bits)
        coded_bits_reshaped = tf.reshape(coded_bits, [batch_size, -1, self.sf])

        # Converte grupos de bits para símbolos decimais
        powers = tf.constant([2**i for i in range(self.sf)], dtype=tf.float32)
        symbols = tf.reduce_sum(tf.cast(coded_bits_reshaped, tf.float32) * powers, axis=-1)
        symbols = tf.cast(symbols, tf.int32)

        # Modulação chirp
        tx_signal = self.chirp_generator(symbols)

        # Flatten para transmissão
        tx_signal_flat = tf.reshape(tx_signal, [batch_size, -1])

        return tx_signal_flat, info_bits, symbols

    def channel(self, tx_signal, ebno_db):
        """
        Modelo de canal com AWGN
        """
        # Calcula potência do sinal
        signal_power = tf.reduce_mean(tf.square(tf.abs(tx_signal)))

        # Adiciona ruído AWGN
        no = ebnodb2no(ebno_db, num_bits_per_symbol=self.sf, coderate=self.fec.rate)
        rx_signal = self.awgn_channel([tx_signal, no])

        return rx_signal

    def receiver(self, rx_signal, symbols_shape, original_info_bits_length):
        """
        Cadeia de recepção LoRa
        """
        batch_size = tf.shape(rx_signal)[0]

        # Reshape para formato de símbolos
        n_symbols = symbols_shape[1]
        n_samples_per_symbol = tf.shape(rx_signal)[1] // n_symbols
        rx_signal_reshaped = tf.reshape(rx_signal, [batch_size, n_symbols, n_samples_per_symbol])

        # Demodulação
        demod_symbols = self.demodulator(rx_signal_reshaped)

        # Converte símbolos de volta para bits
        symbol_bits = []
        for i in range(self.sf):
            bit = tf.bitwise.bitwise_and(tf.right_shift(demod_symbols, i), 1)
            symbol_bits.append(tf.cast(bit, tf.float32))

        demod_bits = tf.stack(symbol_bits, axis=-1)
        demod_bits_flat = tf.reshape(demod_bits, [batch_size, -1])

        # Remove padding se foi adicionado
        total_coded_bits = (original_info_bits_length // self.fec.k) * self.fec.n
        demod_bits_flat = demod_bits_flat[:, :total_coded_bits]

        # Decodificação FEC LoRa
        decoded_bits = self.fec.decode(demod_bits_flat)

        # Ajusta para o tamanho original dos bits de informação
        decoded_bits = decoded_bits[:, :original_info_bits_length]

        return decoded_bits

    def simulate(self, batch_size, ebno_db_range):
        """
        Simula desempenho BER vs SNR
        """
        ber_results = []

        for ebno_db in ebno_db_range:
            # Transmissão
            tx_signal, info_bits, symbols = self.transmitter(batch_size, ebno_db)

            # Canal
            rx_signal = self.channel(tx_signal, ebno_db)

            # Recepção
            original_length = tf.shape(info_bits)[1]
            decoded_bits = self.receiver(rx_signal, tf.shape(symbols), original_length)

            # Calcula BER
            bit_errors = tf.not_equal(info_bits, decoded_bits)
            ber = tf.reduce_mean(tf.cast(bit_errors, tf.float32))
            ber_results.append(ber.numpy())

            print(f"Eb/N0: {ebno_db:.1f} dB, BER: {ber:.2e}")

        return ber_results

### Utilização

In [9]:
# Exemplo de uso
if __name__ == "__main__":
    # Parâmetros LoRa
    sf = 7          # Spreading Factor
    cr = 1          # Code Rate
    bw = 125e3      # Bandwidth 125 kHz
    fs = 1e6        # Sampling frequency 1 MHz

    # Cria modelo LoRa PHY
    lora_phy = LoRaPHY(sf=sf, cr=cr, bw=bw, fs=fs)

    # Simula desempenho
    print("Simulando camada física LoRa...")
    print(f"SF: {sf}, CR: {cr}, BW: {bw/1000:.0f} kHz")
    print(f"Taxa de código: {lora_phy.fec.rate:.2f}")

    ebno_range = np.arange(-10, 5, 2.5)
    batch_size = 1000

    ber_results = lora_phy.simulate(batch_size, ebno_range)

    # Plot resultados
    plt.figure(figsize=(10, 6))
    plt.semilogy(ebno_range, ber_results, 'bo-', label=f'LoRa SF{sf} CR{cr}')
    plt.xlabel('Eb/N0 (dB)')
    plt.ylabel('Bit Error Rate (BER)')
    plt.title(f'Desempenho BER da Camada Física LoRa (SF{sf}, CR{cr})')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()

    print("\nSimulação concluída!")
    print(f"Sensibilidade aproximada (BER=1e-3): {ebno_range[np.argmin(np.abs(np.array(ber_results) - 1e-3)):.1f]} dB")

SyntaxError: invalid decimal literal (ipython-input-9-3916952189.py, line 33)